In [1]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# ============================================================
# STAGE 1: SETUP for H2A
# ============================================================
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate peft
!git clone https://github.com/rfuentesfe/EgyptianHieroglyphicText

import os, cv2, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

DATASET_PATH = '/kaggle/working/EgyptianHieroglyphicText/dataset'
IMG_H, IMG_W = 224, 224
BATCH_SIZE   = 32
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Setup done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
Cloning into 'EgyptianHieroglyphicText'...
remote: Enumerating objects: 20061, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 20061 (delta 4), reused 7 (delta 0), pack-reused 20045 (from 1)
Receiving objects: 100% (20061/20061), 503.06 MiB | 43.71 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Device : cuda
GPU    : Tesla T4
Setup done.


In [2]:
def enhance_contrast(image: Image.Image) -> Image.Image:
    """
    CLAHE contrast enhancement.
    Works on LAB color space to preserve colors while fixing brightness.
    Better than simple histogram equalization.
    """
    img_np = np.clip(np.array(image), 0, 255).astype('uint8')
    lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl     = clahe.apply(l)
    limg   = cv2.merge((cl, a, b))
    result = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    return Image.fromarray(result)


# ── Training transforms (strong augmentation) ───────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.Lambda(enhance_contrast),

    # Geometry augmentations
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5),
    transforms.RandomResizedCrop((IMG_H, IMG_W), scale=(0.8, 1.0)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),

    # Color / texture augmentations
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),

    # Randomly erase a small patch → forces model not to rely on one spot
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

# ── Validation transforms (no augmentation) ─────────────────
val_transforms = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.Lambda(enhance_contrast),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Transforms defined.')

Transforms defined.


In [3]:
class SubsetWithTransform(Dataset):
    """Applies a different transform to a subset of a dataset."""
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=None)
class_names  = full_dataset.classes
num_classes  = len(class_names)
print(f'Classes     : {num_classes}')
print(f'Total images: {len(full_dataset)}')
print(f'Avg per class: {len(full_dataset)/num_classes:.1f}')

n_val   = int(0.2 * len(full_dataset))
n_train = len(full_dataset) - n_val

train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_dataset = SubsetWithTransform(full_dataset, train_indices.indices, train_transforms)
val_dataset   = SubsetWithTransform(full_dataset, val_indices.indices,   val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# Save class mapping for inference / NLP pipeline
with open('class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(full_dataset.class_to_idx, f, ensure_ascii=False, indent=2)

idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}

print(f'\nclass_mapping.json saved')
print(f'Train: {n_train} | Val: {n_val}')

Classes     : 310
Total images: 9703
Avg per class: 31.3

class_mapping.json saved
Train: 7763 | Val: 1940


In [4]:
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    ResNet50 with improved classifier head:
    - Higher dropout (0.5 / 0.3)
    - BatchNorm1d after first linear for stable training
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features  # 2048

    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),            # stabilizes training
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )
    return model


model     = build_model(num_classes, freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,}')
print(f'Total params     : {total:,}')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 198MB/s]


Trainable params : 1,209,142
Total params     : 24,717,174


In [5]:
def mixup_data(x, y, alpha=0.2):
    """Apply MixUp to a batch. Returns mixed images and both label sets."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """MixUp-aware cross entropy loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('MixUp helpers ready.')

MixUp helpers ready.


In [6]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


def run_epoch(model, loader, optimizer, criterion, train=True, use_mixup=False):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0.0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train and use_mixup:
                images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.2)
                outputs = model(images)
                loss    = mixup_criterion(criterion, outputs, y_a, y_b, lam)
                preds   = outputs.argmax(1)
                correct += (lam * (preds == y_a).float() +
                           (1 - lam) * (preds == y_b).float()).sum().item()
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)
                correct += (outputs.argmax(1) == labels).sum().item()

            if train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping — prevents exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total


print('Training loop defined.')

Training loop defined.


In [ ]:
EPOCHS_P1  = 100
PATIENCE_P1 = 10

optimizer_p1 = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-3
)
scheduler_p1 = CosineAnnealingLR(optimizer_p1, T_max=EPOCHS_P1, eta_min=1e-5)

history_p1   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_p1  = 0.0
no_improve   = 0

print('── Phase 1: Head only (backbone frozen) ──')
for epoch in range(EPOCHS_P1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p1, criterion,
                                      train=True,  use_mixup=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   optimizer_p1, criterion,
                                      train=False, use_mixup=False)
    scheduler_p1.step()

    history_p1['train_loss'].append(train_loss)
    history_p1['train_acc'].append(train_acc)
    history_p1['val_loss'].append(val_loss)
    history_p1['val_acc'].append(val_acc)

    print(f'Ep {epoch+1:03d}/{EPOCHS_P1} | '
          f'Loss {train_loss:.3f} | '
          f'Train {train_acc:.3f} | '
          f'Val {val_acc:.3f}', end='')

    if val_acc > best_val_p1:
        best_val_p1 = val_acc
        no_improve  = 0
        torch.save(model.state_dict(), 'best_resnet50_p1.pth')
        print(' ✅')
    else:
        no_improve += 1
        print(f' (no improve {no_improve}/{PATIENCE_P1})')
        if no_improve >= PATIENCE_P1:
            print(f'\n⏹ Early stopping at epoch {epoch+1}')
            break

print(f'\nPhase 1 best val accuracy: {best_val_p1:.3f}')

── Phase 1: Head only (backbone frozen) ──
Ep 001/100 | Loss 4.580 | Train 0.142 | Val 0.220 ✅
Ep 002/100 | Loss 4.213 | Train 0.185 | Val 0.262 ✅
Ep 003/100 | Loss 4.117 | Train 0.198 | Val 0.265 ✅
Ep 004/100 | Loss 4.032 | Train 0.224 | Val 0.288 ✅
Ep 005/100 | Loss 3.973 | Train 0.224 | Val 0.296 ✅
Ep 006/100 | Loss 3.931 | Train 0.228 | Val 0.307 ✅
Ep 007/100 | Loss 3.955 | Train 0.223 | Val 0.306 (no improve 1/20)
Ep 008/100 | Loss 3.889 | Train 0.239 | Val 0.316 ✅
Ep 009/100 | Loss 3.916 | Train 0.232 | Val 0.329 ✅
Ep 010/100 | Loss 3.853 | Train 0.245 | Val 0.325 (no improve 1/20)
Ep 011/100 | Loss 3.807 | Train 0.249 | Val 0.336 ✅
Ep 012/100 | Loss 3.803 | Train 0.257 | Val 0.330 (no improve 1/20)
Ep 013/100 | Loss 3.803 | Train 0.255 | Val 0.341 ✅
Ep 014/100 | Loss 3.784 | Train 0.249 | Val 0.356 ✅
Ep 015/100 | Loss 3.759 | Train 0.267 | Val 0.357 ✅
Ep 016/100 | Loss 3.748 | Train 0.262 | Val 0.343 (no improve 1/20)
Ep 017/100 | Loss 3.731 | Train 0.272 | Val 0.334 (no improve

In [ ]:
# Load best phase 1 checkpoint before unfreezing
model.load_state_dict(torch.load('best_resnet50_p1.pth', map_location=device))
print(f'Loaded Phase 1 best checkpoint (val={best_val_p1:.3f})')

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params now: {trainable:,}')

EPOCHS_P2   = 50
PATIENCE_P2 = 10

optimizer_p2 = AdamW(
    model.parameters(),
    lr=1e-4,           # lower LR — don't destroy pretrained weights
    weight_decay=1e-4
)
scheduler_p2 = CosineAnnealingLR(optimizer_p2, T_max=EPOCHS_P2, eta_min=1e-6)

history_p2  = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_p2 = 0.0
no_improve  = 0

print('\n── Phase 2: Full fine-tune ──')
for epoch in range(EPOCHS_P2):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p2, criterion,
                                      train=True,  use_mixup=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   optimizer_p2, criterion,
                                      train=False, use_mixup=False)
    scheduler_p2.step()

    history_p2['train_loss'].append(train_loss)
    history_p2['train_acc'].append(train_acc)
    history_p2['val_loss'].append(val_loss)
    history_p2['val_acc'].append(val_acc)

    print(f'Ep {epoch+1:03d}/{EPOCHS_P2} | '
          f'Loss {train_loss:.3f} | '
          f'Train {train_acc:.3f} | '
          f'Val {val_acc:.3f}', end='')

    if val_acc > best_val_p2:
        best_val_p2 = val_acc
        no_improve  = 0
        torch.save(model.state_dict(), 'best_resnet50_hieroglyph.pth')
        print(f' ✅ Best saved (val={val_acc:.3f})')
    else:
        no_improve += 1
        print(f' (no improve {no_improve}/{PATIENCE_P2})')
        if no_improve >= PATIENCE_P2:
            print(f'\n⏹ Early stopping at epoch {epoch+1}')
            break

print(f'\nPhase 2 best val accuracy: {best_val_p2:.3f}')

Loaded Phase 1 best checkpoint (val=0.403)
Trainable params now: 24,717,174

── Phase 2: Full fine-tune ──
Ep 001/100 | Loss 2.991 | Train 0.466 | Val 0.630 ✅ Best saved (val=0.630)
Ep 002/100 | Loss 2.674 | Train 0.573 | Val 0.718 ✅ Best saved (val=0.718)
Ep 003/100 | Loss 2.467 | Train 0.635 | Val 0.752 ✅ Best saved (val=0.752)
Ep 004/100 | Loss 2.426 | Train 0.657 | Val 0.777 ✅ Best saved (val=0.777)
Ep 005/100 | Loss 2.206 | Train 0.718 | Val 0.809 ✅ Best saved (val=0.809)
Ep 006/100 | Loss 2.222 | Train 0.718 | Val 0.810 ✅ Best saved (val=0.810)
Ep 007/100 | Loss 2.263 | Train 0.713 | Val 0.811 ✅ Best saved (val=0.811)
Ep 008/100 | Loss 2.104 | Train 0.757 | Val 0.831 ✅ Best saved (val=0.831)
Ep 009/100 | Loss 2.112 | Train 0.761 | Val 0.829 (no improve 1/20)
Ep 010/100 | Loss 2.038 | Train 0.775 | Val 0.840 ✅ Best saved (val=0.840)
Ep 011/100 | Loss 1.962 | Train 0.796 | Val 0.835 (no improve 1/20)
Ep 012/100 | Loss 1.914 | Train 0.806 | Val 0.845 ✅ Best saved (val=0.845)
Ep 013/

In [ ]:
def plot_history(h1, h2):
    train_acc  = h1['train_acc']  + h2['train_acc']
    val_acc    = h1['val_acc']    + h2['val_acc']
    train_loss = h1['train_loss'] + h2['train_loss']
    val_loss   = h1['val_loss']   + h2['val_loss']
    p1_end     = len(h1['train_acc'])
    epochs     = range(1, len(train_acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(epochs, train_acc, label='Train Acc', color='royalblue', linewidth=2)
    axes[0].plot(epochs, val_acc,   label='Val Acc',   color='tomato',    linewidth=2)
    axes[0].axvline(x=p1_end, color='gray', linestyle='--', linewidth=1.5, label='Phase 1 → 2')
    axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Loss plot
    axes[1].plot(epochs, train_loss, label='Train Loss', color='royalblue', linewidth=2)
    axes[1].plot(epochs, val_loss,   label='Val Loss',   color='tomato',    linewidth=2)
    axes[1].axvline(x=p1_end, color='gray', linestyle='--', linewidth=1.5, label='Phase 1 → 2')
    axes[1].set_title('Loss', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(
        f'Training History  |  Best Val: {max(val_acc):.3f}',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_history.png')


plot_history(history_p1, history_p2)

In [ ]:
def load_best_model(path, num_classes):
    m = build_model(num_classes, freeze_backbone=False)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m.to(device)


def predict_sign(image_path: str, model: nn.Module):
    """Single image → Gardiner class + confidence + top3"""
    image  = Image.open(image_path).convert('RGB')
    tensor = val_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)
        top3_probs, top3_idx = torch.topk(probs, 3, dim=1)

    top3 = [
        (idx_to_class[i.item()], p.item())
        for i, p in zip(top3_idx[0], top3_probs[0])
    ]
    return top3[0][0], top3[0][1], top3


# Load the best saved model
best_model = load_best_model('best_resnet50_hieroglyph.pth', num_classes)
print('Best model loaded and ready for inference.')
